# 06 — Avaliação e Golden Set

Conjunto simplificado com **5 clientes** (Etapa 4 do Datathon). Para cada perfil, o serviço/política recomenda um canal e registramos se a decisão faz sentido.



In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.recommend import recommend

golden_path = ROOT / "artifacts" / "golden_set.json"
with open(golden_path, encoding="utf-8") as f:
    golden = json.load(f)

rows = []
for case in golden["cases"]:
    result = recommend(case["customer"], mode="exploit")
    rows.append({
        "case_id": case["case_id"],
        "persona": case["persona"],
        "recommended_offer": result["recommended_offer"],
        "conversion_probability": round(result["conversion_probability"], 4),
        "p_cellular": round(result["conversion_by_arm"]["cellular"], 4),
        "p_telephone": round(result["conversion_by_arm"]["telephone"], 4),
        "expected_offer": case["expected_offer"],
        "makes_sense": result["recommended_offer"] == case["expected_offer"],
        "rationale": case["business_rationale"],
    })

df = pd.DataFrame(rows)
display(df)

print(f"Acertos vs expectativa de negócio: {df['makes_sense'].sum()}/{len(df)}")


## Leitura rápida

O Golden Set não é um teste unitário rígido de ML: valida se a política adaptativa recomenda canais coerentes com o aprendizado do S2 (preferência por `cellular`, maior conversão esperada) e com o contexto de cada persona.

